In [1]:
using BSON: @load
using CSV
using Dates
using DriftDiffusionModels
using HiddenMarkovModels
using MultivariateStats
using LinearAlgebra
using Clustering
using DataFrames
using HypothesisTests
using Distances
using Random
using StatsBase
using StatsPlots
using PlotUtils
using Plots
using Turing
using MCMCChains
using CategoricalArrays
using Base.Threads: @threads
using CairoMakie

## Preprocessing

In [2]:
# set data dirtectory
data_dir = joinpath(@__DIR__, "..", "data")

# read out all BSOn files that have a 4 in them (K=4)
bson_files = filter(f -> occursin("K4", f), readdir(data_dir; join=true))

# read all data into a bson file. We need to define the following struct for it to work right.
struct DDMHMMFit
    hmm::PriorHMM
    logL::Float64
    logL_evolution::Vector{Float64}
end

ddmhmm_fits = DDMHMMFit[]  

for bson_file in bson_files
    @load bson_file fit   # defines `fit` in this scope
    push!(ddmhmm_fits, fit)
end

In [3]:
ddmhmm_fits

18-element Vector{DDMHMMFit}:
 DDMHMMFit(PriorHMM{Float64, DriftDiffusionModel}([0.4470226731912877, 0.24418872400797034, 0.3087886028007418, 1.7408329146292395e-16], [0.7731229647375347 0.04226027213105377 0.18350631600748177 0.001110447123929687; 0.04364951660243847 0.9483084114450056 0.008042071952549672 6.448159890273335e-15; 0.3119494473618833 0.010572103399372269 0.5575737348382933 0.11990471440045108; 0.003411443463936616 3.1637337541509816e-8 0.32900788323116065 0.6675806416675653], DriftDiffusionModel[DriftDiffusionModel(1.4700214601398267, 0.8008726178226451, 0.29232052626778426, 0.7548477788783509), DriftDiffusionModel(1.8118013067009697, 0.7258178544872764, 0.3581318331129916, 0.8220889167332065), DriftDiffusionModel(1.147888611095051, 0.5567456593869297, 0.3467791323055344, 0.6517063654066835), DriftDiffusionModel(1.6935745851406607, 0.6122746710121455, 0.3029767916821212, 0.518806389622618)], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0, 1.0,

In [4]:
ddm_params = Matrix{Float64}(undef, 4, 18 * 4) # 4 parameters per DDM, (18 subjects x 4 states)

for (i, fit) in enumerate(ddmhmm_fits)
    for k in 1:4
        ddm = fit.hmm.dists[k]
        ddm_params[:, (i - 1) * 4 + k] = [ddm.v, ddm.B, ddm.a₀, ddm.τ]
    end
end

ddm_params_df = DataFrame(ddm_params', [:v, :B, :a0, :tau])
animal_files = [split(fname, "\\")[end] for fname in bson_files]
animal_name  = [split(file, "_")[1] for file in animal_files]

ddm_params_df.rat = repeat(animal_name, inner=4);

ddm_params_clean = Matrix(select(ddm_params_df, Not(:rat))) # for numerical analyses

# standardize first if need later
zscore(x; dims) = (x .- mean(x; dims=dims)) ./ std(x; dims=dims)
ddm_params_z = zscore(ddm_params_clean'; dims=2)

4×72 Matrix{Float64}:
  0.0884106   0.0123428  -0.159011  …  -0.623018    -0.135475   -0.299235
 -0.329032   -0.153295   -0.494668      0.00684298  -0.333599   -0.473151
 -1.05581    -0.630828   -0.70414      -0.186832    -0.236163    0.290069
  0.833301    1.09977     0.424557     -2.15416     -0.0134415  -0.466619

## Compare Parameters between models

In [5]:
accs = Vector{Float64}(undef, size(ddm_params_df, 1))
mean_rt = Vector{Float64}(undef, size(ddm_params_df, 1))
for (idx, i) in enumerate(eachrow(ddm_params_df))
    ddm = DriftDiffusionModels.DriftDiffusionModel(v=i.v, B=i.B, a₀=i.a0, τ=i.tau)
    data = simulateDDM(ddm, 10000)
    is_correct = [d.s == d.choice for d in data]
    rts = [d.rt for d in data]
    accs[idx] = mean(is_correct)
    mean_rt[idx] = mean(rts)
end
ddm_params_df.acc = accs;
ddm_params_df.mean_rt = mean_rt;

In [6]:

function high_low_by_rat(df::DataFrame; acc::Symbol=:acc)
    g = groupby(df, :rat)

    high = combine(g) do sdf
        sdf[argmax(sdf[!, acc]), :]
    end

    low = combine(g) do sdf
        sdf[argmin(sdf[!, acc]), :]
    end

    rename!(high, Symbol.(names(high)) .=> Symbol.(string.(names(high)) .* "_high"))
    rename!(low,  Symbol.(names(low))  .=> Symbol.(string.(names(low))  .* "_low"))

    return innerjoin(high, low, on = :rat_high => :rat_low)
end

hl = high_low_by_rat(ddm_params_df)

hl.Δv   = hl.v_high   .- hl.v_low
hl.ΔB   = hl.B_high   .- hl.B_low
hl.Δa0  = hl.a0_high  .- hl.a0_low
hl.Δτ   = hl.tau_high .- hl.tau_low

function paired_spaghetti(hl::DataFrame, low::Symbol, high::Symbol; title="")
    lo = hl[!, low]
    hi = hl[!, high]
    n = length(lo)

    p = plot(size =(500,500), xticks=([1,2], ["Low", "High"]), title=title, legend=false)
    for i in 1:n
        plot!(p, [1,2], [lo[i], hi[i]], lw=1, alpha=0.6)
        scatter!(p, [1,2], [lo[i], hi[i]], alpha=0.8, ms=4)
    end

    return p
end

delta_tau = paired_spaghetti(hl, :tau_low, :tau_high; title="τ")
delta_a0  = paired_spaghetti(hl, :a0_low,  :a0_high;  title="a₀")
delta_B   = paired_spaghetti(hl, :B_low,   :B_high;   title="B")
delta_v   = paired_spaghetti(hl, :v_low,   :v_high;   title="v")

param_diff_plot = plot(delta_v, delta_B, delta_a0, delta_tau; layout=(2,2))
# savefig(param_diff_plot, "../results/param_diff_plot_population.svg")

UndefVarError: UndefVarError: `plot` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in GR.jlgr.
    - Also exported by GR.
Hint: a global variable of this name also exists in RecipesBase.
    - Also exported by Plots.
    - Also exported by StatsPlots.
Hint: a global variable of this name also exists in Makie.
    - Also exported by CairoMakie.

In [7]:
function paired_tests(hl::DataFrame, Δ::Symbol)
    x = skipmissing(hl[!, Δ]) |> collect

    t  = OneSampleTTest(x)      # mean(Δ) = 0
    w  = SignedRankTest(x)      # median(Δ) = 0 (robust default)

    return (
        n = length(x),
        mean = mean(x),
        median = median(x),
        t_p = pvalue(t),
        w_p = pvalue(w),
    )
end

paired_tests(hl, :Δv)

(n = 18, mean = 0.9970279075701297, median = 0.5415665189843006, t_p = 0.03163188543742542, w_p = 7.62939453125e-6)

In [8]:
hl.bias_best = hl.a0_high .- 0.5
hl.bias_2worst  = hl.a0_low  .- 0.5
SignedRankTest(hl.bias_best) |> pvalue
SignedRankTest(hl.bias_2worst)  |> pvalue

0.6705245971679688

## Within-Animal SAT

In [9]:
# SAT curvs
function center_within_animal(df::DataFrame; rat::Symbol=:rat, rt::Symbol=:mean_rt, acc::Symbol=:mean_acc) 
    g = groupby(df, rat) 
    return DataFrames.transform(g, rt => (x -> x .- mean(x)) => :rt_c, acc => (x -> x .- mean(x)) => :acc_c, ) 
end 

function add_acc_rank(df::DataFrame; rat::Symbol=:rat, acc::Symbol=:acc_c, rev::Bool=false)
    g = groupby(df, rat)
    return DataFrames.transform(g, acc => (x -> invperm(sortperm(x; rev=rev))) => :acc_rank)
end

function add_rt_order(df::DataFrame; rat::Symbol=:rat, rt::Symbol=:rt_c)
    g = groupby(df, rat)
    return DataFrames.transform(g, rt => (x -> invperm(sortperm(x))) => :rt_order)
end

function plot_sat_dots_by_rat_acc_rank(df::DataFrame;
    rt::Symbol=:rt_c,
    acc::Symbol=:acc_c,
    rat::Symbol=:rat,
    acc_rank::Symbol=:acc_rank,
    title::String="Within-animal SAT",
    markersize::Real=4,
    alpha::Real=0.65,
)
    K = maximum(df[!, acc_rank])
    pal = palette(:auto) 
    palK = [pal[mod1(i, length(pal))] for i in 1:K]         # vector of Colors

    p = scatter(
        title=title,
        xlabel="Mean Centered RT (s)",
        ylabel="Mean Centered Accuracy",
        legend=false,
        fontfamily=:helvetica
    )

    for sdf in groupby(df, rat)
        r = sdf[!, acc_rank]
        cols = palK[r]  # rank -> default Julia/Plots color

        scatter!(p, sdf[!, rt], sdf[!, acc];
            markersize=markersize,
            alpha=alpha,
            markercolor=cols,
            markerstrokewidth=0
        )
    end

    return p
end

function overlay_mean_sat_rtbin!(p, df::DataFrame;
    rt::Symbol=:rt_c,
    acc::Symbol=:acc_c,
    order::Symbol=:rt_order,
    line_lw::Real=5,
    ribbon_alpha::Real=0.15
)
    summ = combine(groupby(df, order),
        rt  => mean => :rt_mean,
        acc => mean => :acc_mean,
        acc => (x -> std(x)/sqrt(length(x))) => :acc_sem
    )
    sort!(summ, order)

    # black line + SEM ribbon
    plot!(p, summ.rt_mean, summ.acc_mean; lw=line_lw, color=:black, alpha=1.0)
    plot!(p, summ.rt_mean, summ.acc_mean; ribbon=summ.acc_sem, fillalpha=ribbon_alpha, color=:black, lw=0)

    return p
end

"""
Compute per-(rat,state) mean of centered RT/ACC, plus a within-rat accuracy rank for coloring.
Returns a small DataFrame with columns:
  rat, state, rt_state_mean, acc_state_mean, acc_rank
"""
function state_centers(df::DataFrame;
    rat::Symbol=:rat,
    state::Symbol=:state,
    rt::Symbol=:rt_c,
    acc::Symbol=:acc_c,
    rev::Bool=false
)
    # 1) state means in centered coordinates
    c = combine(groupby(df, [state]),
        rt  => mean => :rt_state_mean,
        acc => mean => :acc_state_mean,
    )


    return c
end

"""
Overlay state-center points on an existing SAT scatter plot.
Colors use the same palette logic as your dot plot (rank->color).
"""
function overlay_state_centers!(p, df::DataFrame;
    rat::Symbol=:rat,
    state::Symbol=:state,
    rt::Symbol=:rt_c,
    acc::Symbol=:acc_c,
    rev::Bool=false,
    markersize::Real=9,
    alpha::Real=0.95,
    marker=:diamond,
    stroke_color=:black,
    stroke_width::Real=1.0
)
    c = state_centers(df; rat=rat, state=state, rt=rt, acc=acc, rev=rev)

    K = maximum(c.acc_rank)
    pal = palette(:auto)
    palK = [pal[mod1(i, length(pal))] for i in 1:K]

    # If you want centers plotted per rat "on top" but not re-grouped visually,
    # a single scatter! call is fine.
    cols = palK[c.acc_rank]

    scatter!(p, c.rt_state_mean, c.acc_state_mean;
        markersize=markersize,
        alpha=alpha,
        marker=marker,
        markercolor=cols,
        markerstrokecolor=stroke_color,
        markerstrokewidth=stroke_width,
        label=false
    )

    return p, c
end

dfc  = center_within_animal(ddm_params_df; rat=:rat, rt=:mean_rt, acc=:acc)
dfc  = add_rt_order(dfc; rat=:rat, rt=:rt_c)
dfc  = add_acc_rank(dfc; rat=:rat, acc=:acc_c, rev=false)

p = plot_sat_dots_by_rat_acc_rank(dfc; rt=:rt_c, acc=:acc_c, rat=:rat, acc_rank=:acc_rank)

# Overlay per-state centers
# p, centers = overlay_state_centers!(p, dfc; rat=:rat, state=:acc_rank, rt=:rt_c, acc=:acc_c, rev=false)



# savefig(p, "../results/population_sat_rtLine_accColorDots.svg")


UndefVarError: UndefVarError: `scatter` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in Plots.
    - Also exported by StatsPlots.
Hint: a global variable of this name also exists in GR.jlgr.
    - Also exported by GR.
Hint: a global variable of this name also exists in Makie.
    - Also exported by CairoMakie.

## Calculate Regression between model parameters and accuracy

In [10]:
@model function parameter_regression(acc, ddm_params)
    N, P = size(ddm_params)

    β ~ MvNormal(zeros(P), I)
    ϕ ~ Exponential(1.0)

    η = ddm_params * β
    p = logistic.(η)       # mean in (0,1)

    ϵ = 1e-6
    p = clamp.(p, ϵ, 1 - ϵ)

    for n in 1:N
        a = p[n] * ϕ
        b = (1 - p[n]) * ϕ
        acc[n] ~ Beta(a, b)
    end
end

ddm_params_reg = select(ddm_params_df, [:v, :B, :a0, :tau]) |> Matrix
ddm_params_reg = zscore(ddm_params_reg; dims=1)
ddm_params_reg = hcat(ddm_params_reg, ones(size(ddm_params_reg, 1)))  # add intercept


acc = ddm_params_df.acc |> collect
acc = clamp.(acc, 1e-6, 1 - 1e-6)

model = parameter_regression(acc, ddm_params_reg)
chain = sample(model, NUTS(0.65), 5000; progress=true)

Sampling   0%|█                                         |  ETA: N/A
┌ Info: Found initial step size
│   ϵ = 0.4
└ @ Turing.Inference C:\Users\senne\.julia\packages\Turing\lmRwJ\src\mcmc\hmc.jl:220
Sampling   0%|█                                         |  ETA: 0:29:12
Sampling   1%|█                                         |  ETA: 0:14:35
Sampling   2%|█                                         |  ETA: 0:09:42
Sampling   2%|█                                         |  ETA: 0:07:15
Sampling   2%|██                                        |  ETA: 0:05:47
Sampling   3%|██                                        |  ETA: 0:04:49
Sampling   4%|██                                        |  ETA: 0:04:07
Sampling   4%|██                                        |  ETA: 0:03:35
Sampling   4%|██                                        |  ETA: 0:03:11
Sampling   5%|███                                       |  ETA: 0:02:51
Sampling   6%|███                                       |  ETA: 0:02:35
Sampling   

Chains MCMC chain (5000×20×1 Array{Float64, 3}):

Iterations        = 1001:1:6000
Number of chains  = 1
Samples per chain = 5000
Wall duration     = 12.67 seconds
Compute duration  = 12.67 seconds
parameters        = β[1], β[2], β[3], β[4], β[5], ϕ
internals         = n_steps, is_accept, acceptance_rate, log_density, hamiltonian_energy, hamiltonian_energy_error, max_hamiltonian_energy_error, tree_depth, numerical_error, step_size, nom_step_size, lp, logprior, loglikelihood

Use `describe(chains)` for summary statistics and quantiles.


In [11]:
N = length(acc)
acc_miss = Vector{Union{Missing, Float64}}(missing, N)

pp_model = parameter_regression(acc_miss, ddm_params_reg)
pp = predict(pp_model, chain)

mean_acc = Vector{Float64}(undef, N)
for n in 1:N
    mean_acc[n] = mean(pp[Symbol("acc[$n]")])
end

In [12]:
N = length(acc)
ord = sortperm(acc)                 # increasing acc
acc_rank = similar(ord)             # Int vector
acc_rank[ord] .= 1:N                # inverse permutation => ranks 1..N

K = 4
acc_bin = clamp.(ceil.(Int, acc_rank .* K ./ N), 1, K)  # 1..K

pal = palette(:auto)
palK = [pal[mod1(i, length(pal))] for i in 1:K]
pt_colors = palK[acc_bin]

scat_plot = scatter(acc, mean_acc;
    xlabel="Observed accuracy",
    ylabel="Predicted accuracy",
    fontfamily="helvetica",
    aspect_ratio=:equal,
    xlims=(0.5, 1.0),
    ylims=(0.5, 1.0),
    markercolor=pt_colors,
    markersize=4,
    alpha=0.75,
    markerstrokewidth=0,
    label=""
)

plot!(scat_plot, [0.5, 1.0], [0.5, 1.0], lc=:red, ls=:dash, label="")
savefig(scat_plot, "../results/acc_pred_population_rankcolored.svg")


UndefVarError: UndefVarError: `scatter` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in Plots.
    - Also exported by StatsPlots.
Hint: a global variable of this name also exists in GR.jlgr.
    - Also exported by GR.
Hint: a global variable of this name also exists in Makie.
    - Also exported by CairoMakie.

## Regression of Model Parameters and RTs

In [13]:
@model function parameter_regression_rt(rt::AbstractVector, ddm_params::AbstractMatrix)
    N, P = size(ddm_params)

    # Priors
    β ~ MvNormal(zeros(P), Matrix{Float64}(I, P, P))
    α ~ LogNormal(0.0, 0.5)              # Gamma shape > 0 (controls variance)

    # Log link: mean RT
    μ = exp.(ddm_params * β)             # length N, all positive

    # Likelihood: Gamma with mean μ_i  (Gamma(shape, scale))
    @inbounds for i in 1:N
        rt[i] ~ Gamma(α, μ[i] / α)
    end
end

parameter_regression_rt (generic function with 2 methods)

In [14]:
model_rt = parameter_regression_rt(ddm_params_df.mean_rt |> collect, ddm_params_reg)
chain_rt = sample(model_rt, NUTS(0.65), 5000)

Sampling   0%|█                                         |  ETA: N/A
┌ Info: Found initial step size
│   ϵ = 0.4
└ @ Turing.Inference C:\Users\senne\.julia\packages\Turing\lmRwJ\src\mcmc\hmc.jl:220
Sampling   0%|█                                         |  ETA: 0:15:03
Sampling   1%|█                                         |  ETA: 0:07:30
Sampling   2%|█                                         |  ETA: 0:05:00
Sampling   2%|█                                         |  ETA: 0:03:44
Sampling   2%|██                                        |  ETA: 0:02:59
Sampling   3%|██                                        |  ETA: 0:02:29
Sampling   4%|██                                        |  ETA: 0:02:08
Sampling   4%|██                                        |  ETA: 0:01:51
Sampling   4%|██                                        |  ETA: 0:01:39
Sampling   5%|███                                       |  ETA: 0:01:28
Sampling   6%|███                                       |  ETA: 0:01:21
Sampling   

Chains MCMC chain (5000×20×1 Array{Float64, 3}):

Iterations        = 1001:1:6000
Number of chains  = 1
Samples per chain = 5000
Wall duration     = 6.91 seconds
Compute duration  = 6.91 seconds
parameters        = β[1], β[2], β[3], β[4], β[5], α
internals         = n_steps, is_accept, acceptance_rate, log_density, hamiltonian_energy, hamiltonian_energy_error, max_hamiltonian_energy_error, tree_depth, numerical_error, step_size, nom_step_size, lp, logprior, loglikelihood

Use `describe(chains)` for summary statistics and quantiles.


In [15]:
describe(chain_rt)

Chains MCMC chain (5000×20×1 Array{Float64, 3}):

Iterations        = 1001:1:6000
Number of chains  = 1
Samples per chain = 5000
Wall duration     = 6.91 seconds
Compute duration  = 6.91 seconds
parameters        = β[1], β[2], β[3], β[4], β[5], α
internals         = n_steps, is_accept, acceptance_rate, log_density, hamiltonian_energy, hamiltonian_energy_error, max_hamiltonian_energy_error, tree_depth, numerical_error, step_size, nom_step_size, lp, logprior, loglikelihood

Summary Statistics

  parameters      mean       std      mcse    ess_bulk    ess_tail      rhat   ⋯
      Symbol   Float64   Float64   Float64     Float64     Float64   Float64   ⋯

        β[1]   -0.4702    0.0957    0.0019   2492.8118   2345.9983    1.0009   ⋯
        β[2]    0.5440    0.1036    0.0021   2388.8892   1841.7377    1.0003   ⋯
        β[3]   -0.1181    0.0466    0.0006   5412.3175   3664.0990    1.0002   ⋯
        β[4]    0.2052    0.0497    0.0008   3832.4620   3459.5135    1.0002   ⋯
        β[5]    

In [16]:
using StatsPlots
using Statistics
using ColorSchemes   # if you use palette(:auto) already, keep that
# using StatsBase    # optional, not required

# --- observed RT vector ---
rt = ddm_params_df.mean_rt |> collect
# Gamma requires strictly positive
rt = clamp.(rt, 1e-9, Inf)

# --- posterior predictive draws ---
N = length(rt)
rt_miss = Vector{Union{Missing, Float64}}(missing, N)

pp_model_rt = parameter_regression_rt(rt_miss, ddm_params_reg)
pp_rt = predict(pp_model_rt, chain_rt)

# --- predicted mean RT per observation (posterior predictive mean) ---
mean_rt_pred = Vector{Float64}(undef, N)
for i in 1:N
    mean_rt_pred[i] = mean(pp_rt[Symbol("rt[$i]")])
end

# --- rank-based color bins (like your acc plot) ---
ord = sortperm(rt)         # increasing RT
rt_rank = similar(ord)
rt_rank[ord] .= 1:N

K = 4
rt_bin = clamp.(ceil.(Int, rt_rank .* K ./ N), 1, K)

pal = palette(:auto)
palK = [pal[mod1(i, length(pal))] for i in 1:K]
pt_colors = palK[rt_bin]

# --- scatter: observed vs predicted ---
scat_rt = scatter(
    rt, mean_rt_pred;
    xlabel = "Observed mean RT",
    ylabel = "Predicted mean RT (posterior predictive)",
    fontfamily = "helvetica",
    markercolor = pt_colors,
    markersize = 4,
    alpha = 0.75,
    markerstrokewidth = 0,
    label = ""
)

# identity line (y=x) over the plotted range
xmin, xmax = extrema(rt)
ymin, ymax = extrema(mean_rt_pred)
lo = min(xmin, ymin); hi = max(xmax, ymax)
plot!(scat_rt, [lo, hi], [lo, hi], lc=:red, ls=:dash, label="")

savefig(scat_rt, "../results/rt_pred_population_rankcolored.svg")

# --- OPTIONAL: log-scale version (often better for RT) ---
scat_rt_log = scatter(
    rt, mean_rt_pred;
    xscale = :log10,
    yscale = :log10,
    xlabel = "Observed mean RT (log10)",
    ylabel = "Predicted mean RT (log10)",
    fontfamily = "helvetica",
    markercolor = pt_colors,
    markersize = 4,
    alpha = 0.75,
    markerstrokewidth = 0,
    label = ""
)
plot!(scat_rt_log, [lo, hi], [lo, hi], lc=:red, ls=:dash, label="")
savefig(scat_rt_log, "../results/rt_pred_population_rankcolored_log.svg")


ArgumentError: ArgumentError: Package ColorSchemes not found in current path.
- Run `import Pkg; Pkg.add("ColorSchemes")` to install the ColorSchemes package.

In [17]:
β_raw = Matrix{Float64}(undef, 5000, 5)  # 4 params + intercept
for i in 1:5
    β_raw[:, i] = chain[Symbol("β[$i]")]
end

beta_df = DataFrame(β_raw, [:v, :B, :a0, :tau, :intercept])
beta_long = stack(beta_df; variable_name=:param, value_name=:beta)

summ = combine(groupby(beta_long, :param),
    :beta => mean => :mean,
    :beta => (x -> quantile(x, 0.025)) => :lo,
    :beta => (x -> quantile(x, 0.975)) => :hi
)

summ.param = string.(summ.param)
order = ["v", "B", "a0", "tau", "intercept"]

# reorder rows according to order
summ = leftjoin(DataFrame(param=order, idx=1:length(order)), summ, on=:param)
sort!(summ, :idx)

y = 1:nrow(summ)
forest_plot = plot()
scatter!(summ.mean, y;
    fontfamily="helvetica",
    xerror = (summ.mean .- summ.lo, summ.hi .- summ.mean),
    yticks = (y, summ.param),
    xlabel = "βᵢ",
    ylabel = "",
    legend = false,
    markersize = 6
)

vline!([0.0], ls=:dash, label="")

savefig(forest_plot, "../results/beta_forest_population.svg")


UndefVarError: UndefVarError: `plot` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in GR.jlgr.
    - Also exported by GR.
Hint: a global variable of this name also exists in RecipesBase.
    - Also exported by Plots.
    - Also exported by StatsPlots.
Hint: a global variable of this name also exists in Makie.
    - Also exported by CairoMakie.

In [ ]:
const _dt_formats = (
    dateformat"yyyy-mm-dd HH:MM:SS.s",
    dateformat"yyyy-mm-dd HH:MM:SS",
    dateformat"yyyy-mm-ddTHH:MM:SS.s",
    dateformat"yyyy-mm-ddTHH:MM:SS",
)

parse_dt(x) = x isa DateTime ? x :
              x isa Date     ? DateTime(x) :
              begin
                  s = String(x)
                  for df in _dt_formats
                      try
                          return DateTime(s, df)
                      catch
                      end
                  end
                  error("Couldn't parse trial_datetime string: $x. Add your format to _dt_formats.")
              end

function load_rat_data(rat::AbstractString;
    data_file = joinpath(@__DIR__, "..", "data", "processed_rat_data.csv.gz"),
)

    # Read in and structure data
    rat_df = CSV.read(data_file, DataFrame)
    rat_df = rat_df[rat_df.daily .== "24 hr", :] # only keep 24 hour data

    # preprocess the data to have numerics
    replace!(rat_df[!, :choose_right], 0 => -1)

    mapping = Dict("right" => 1, "left" => -1)
    DataFrames.transform!(rat_df,
        :correct_side => ByRow(cs -> get(mapping, cs, missing)) => :correct_side_numeric
    )

    rat_of_interest = rat_df[rat_df.name .== rat, :]

    # parse trial dates
    dts = parse_dt.(rat_of_interest.trial_datetime)
    dates = Date.(dts)

    unique_dates = sort(unique(dates))

    # Create a vector of vectors, where each inner vector contains DDMResults for one day
    results_by_date = Vector{Vector{DDMResult}}()

    for date in unique_dates
        day_indices = findall(dates .== date)
        isempty(day_indices) && continue

        day_rts       = rat_of_interest.rt[day_indices]
        day_outcomes  = rat_of_interest.choose_right[day_indices]
        day_stim_side = rat_of_interest.correct_side_numeric[day_indices]

        day_results = [DDMResult(rt, choice, stim)
                       for (rt, choice, stim) in zip(day_rts, day_outcomes, day_stim_side)]

        push!(results_by_date, day_results)
    end

    all_results = vcat(results_by_date...)
    seq_ends = cumsum(length.(results_by_date))

    return rat_of_interest, dates, unique_dates, results_by_date, all_results, seq_ends
end

function state_occupancy_by_hour(
    hmm,
    γ,
    rat_of_interest,
    dates,
    unique_dates;
    light_on_hour = 7.5,
    K = size(γ,1),
)

    # Rebuild concatenation index (must match γ)
    idxs_by_date = Vector{Vector{Int}}()
    for date in unique_dates
        push!(idxs_by_date, findall(dates .== date))
    end
    concat_idxs = reduce(vcat, idxs_by_date)

    # Parse datetimes
    trial_dt_col = rat_of_interest.trial_datetime
    hours_concat = [
        mod(hour(parse_dt(trial_dt_col[i])) - light_on_hour, 24)
        for i in concat_idxs
    ]

    occ = zeros(24, K)
    n_per_hour = zeros(Int, 24)

    for h in 0:23
        idx_h = findall(x -> floor(Int, x) == h, hours_concat)
        n = length(idx_h)
        n_per_hour[h+1] = n
        if n > 0
            occ[h+1, :] .= vec(mean(γ[:, idx_h]; dims=2))
        else
            occ[h+1, :] .= NaN
        end
    end

    return occ, n_per_hour
end


function state_accuracy(γ, all_results)
    correct = [r.choice == r.s for r in all_results]
    K = size(γ,1)
    acc = zeros(K)
    for k in 1:K
        w = γ[k, :]
        acc[k] = sum(w .* correct) / sum(w)
    end
    return acc
end

K = 4
n_animals = length(bson_files)

# store (animal × hour × accuracy-rank)
occ_by_rank = Array{Float64}(undef, n_animals, 24, K)

for (i, rat) in enumerate(animal_name)

    # load fit 
    path = joinpath(data_dir, rat * "_K4_best.bson")
    @load path fit
    hmm = fit.hmm

    # load & preprocess data 
    rat_of_interest, dates, unique_dates, results_by_date, all_results, seq_ends = load_rat_data(rat)

    # posterior 
    γ, _ = forward_backward(hmm, all_results; seq_ends=seq_ends)

    # aliggn by accuracy
    acc = state_accuracy(γ, all_results)
    order = sortperm(acc)  # lowest → highest

    # state occ
    occ, _ = state_occupancy_by_hour(
        hmm, γ, rat_of_interest, dates, unique_dates
    )

    # reorder states by accuracy
    occ_by_rank[i, :, :] .= occ[:, order]
end


In [ ]:
params = [:v, :B, :a0, :tau]
long = stack(dfc, params; variable_name=:param, value_name=:value)

long.acc_rank = categorical(long.acc_rank; ordered=true)

p = @df long boxplot(
    :acc_rank, :value;
    group=:param,
    layout=(2,2),
    legend=false,
    xlabel="acc_rank",
    ylabel="parameter value",
    outliers=false,       
    whisker_width=0.6,
    fontfamily="helvetica"
)

# overlay swarm/jitter points on the same axes
# jittered points on top
@df long dotplot!(
    :acc_rank, :value;
    group=:param,
    layout=(2,2),
    legend=false,
    jitter=0.25,     # horizontal jitter amount
    alpha=0.45,
    markersize=3
)

# remove outliers that distort axes
plot!(p[1], ylims=(0, 6))
plot!(p[4], ylims=(0, 1.5))

display(p)

savefig(p, "../results/parameter_boxswarm_by_accRank_population.svg")


In [ ]:
labels = ["Lowest accuracy", "Low-mid accuracy", "High-mid accuracy", "Highest accuracy"]

p = plot(link=:x, legend=:topleft, fontfamily="helvetica")  

for k in 1:K
    plot!(p, hours, mean_occ[:, k],
        ribbon = sem_occ[:, k],
        lw = 2,
        label = labels[k],
    )
end

xlabel!(p, "Hour from lights on (binned)")
ylabel!(p, "State occupancy")
title!(p, "Population State Occupancy ")
p


savefig(p, "../results/population_state_occupancy.svg")

In [ ]:
function enrichment_per_rat(occ_by_rank, hour_mask)
    n_animals, H, K = size(occ_by_rank)
    enrich = fill(NaN, n_animals, K)
    for i in 1:n_animals, k in 1:K
        x_all = occ_by_rank[i, :, k]
        x_win = occ_by_rank[i, hour_mask, k]
        x_all = x_all[.!isnan.(x_all)]
        x_win = x_win[.!isnan.(x_win)]
        enrich[i, k] = mean(x_win) / mean(x_all)
    end
    return enrich
end

dark_mask = falses(24); dark_mask[13:24] .= true
feeding_mask = falses(24); feeding_mask[7:9] .= true

enrich_dark = enrichment_per_rat(occ_by_rank, dark_mask)
enrich_feed = enrichment_per_rat(occ_by_rank, feeding_mask)

In [ ]:
function mean_sem(x)
    x = x[.!isnan.(x)]
    return mean(x), std(x)/sqrt(length(x))
end

m_dark = [mean_sem(enrich_dark[:,k])[1] for k in 1:K]
s_dark = [mean_sem(enrich_dark[:,k])[2] for k in 1:K]

m_feed = [mean_sem(enrich_feed[:,k])[1] for k in 1:K]
s_feed = [mean_sem(enrich_feed[:,k])[2] for k in 1:K]

p1 = bar(labels, m_dark, yerror=s_dark, legend=false, title="Dark-cycle enrichment", ylabel="Enrichment (× baseline)")
hline!(p1, [1.0], alpha=0.4)

p2 = bar(labels, m_feed, yerror=s_feed, legend=false, title="Feeding-time enrichment", ylabel="Enrichment (× baseline)")
hline!(p2, [1.0], alpha=0.4)

p = plot(p1, p2, link=:y, layout=(1,2), size=(600, 300), fontfamily="helvetica")

savefig(p, "../results/population_enrichment.svg")

# Population Parameter Plots

In [ ]:
# this is slow but works
function add_state_rank_by_acc!(df::DataFrame;
    rat::Symbol=:rat,
    acc::Symbol=:acc,
    state::Symbol=:state,
    ascending::Bool=true,   # true: 1 = lowest acc, 4 = highest acc
)
    transform!(groupby(df, rat)) do sdf
        ord = sortperm(sdf[!, acc]; rev = !ascending)  # indices in sorted order
        ranks = similar(ord)
        ranks[ord] = 1:length(ord)                    # map back to original row order
        return DataFrame(state => ranks)
    end
    return df
end

add_state_rank_by_acc!(ddm_params_df; rat=:rat, acc=:acc, state=:state, ascending=true)

# Robust color limits so one outlier doesn't dominate the color scale.
# q=(0.05,0.95) means clip the bottom/top 5%.
function robust_clims(M; q=(0.05, 0.95))
    v = vec(M)
    v = v[isfinite.(v)]
    lo, hi = quantile(v, q)
    if !(isfinite(lo) && isfinite(hi)) || lo == hi
        lo, hi = minimum(v), maximum(v)
    end
    return (lo, hi)
end

function rats_order_from_param(df::DataFrame, param::Symbol;
    rat::Symbol=:rat,
    state::Symbol=:state,
    order_rows_by::Symbol=:mean,   # :mean or :median
    order_rows_rev::Bool=true,     # true => highest on top (with yflip=true)
)
    d = select(df, rat, state, param)

    rats0  = unique(string.(d[!, rat]))
    states = sort(unique(d[!, state]))

    rat_to_i = Dict(rats0[i] => i for i in eachindex(rats0))
    state_to_j = Dict(states[j] => j for j in eachindex(states))

    M0 = fill(NaN, length(rats0), length(states))
    for r in eachrow(d)
        M0[rat_to_i[string(r[rat])], state_to_j[r[state]]] = r[param]
    end

    rowstat = Vector{Float64}(undef, size(M0, 1))
    for i in 1:size(M0, 1)
        v = M0[i, :]
        v = v[isfinite.(v)]
        rowstat[i] = isempty(v) ? NaN :
            (order_rows_by === :median ? median(v) : mean(v))
    end

    idx_valid = findall(isfinite, rowstat)
    idx_nan   = findall(x -> !isfinite(x), rowstat)

    idx_sorted = idx_valid[sortperm(rowstat[idx_valid]; rev=order_rows_rev)]
    idx = vcat(idx_sorted, idx_nan)

    return rats0[idx]  # Vector{String}
end

function param_heatmap_pretty_makie(df::DataFrame, param::Symbol;
    rat::Symbol=:rat,
    state::Symbol=:state,
    title::String="$(param) by rat × state",
    q=(0.05, 0.95),
    rats_order::Union{Nothing,Vector{String}}=nothing,
    cell_px::Int=26,
    left_pad::Int=140,
    bottom_pad::Int=90,
    top_pad::Int=70,
    right_pad::Int=40,
    cmap=:viridis,
)

    d = select(df, rat, state, param)

    # columns (states)
    states = sort(unique(d[!, state]))
    state_to_j = Dict(states[j] => j for j in eachindex(states))

    # rows (rats)
    rats0 = unique(string.(d[!, rat]))
    rats = if rats_order === nothing
        rats0
    else
        present = Set(rats0)
        ordered = [r for r in rats_order if r in present]
        extras  = [r for r in rats0 if !(r in Set(ordered))]
        vcat(ordered, extras)
    end
    rat_to_i = Dict(rats[i] => i for i in eachindex(rats))

    # M: nrows(rats) × ncols(states)
    M = fill(NaN, length(rats), length(states))
    for r in eachrow(d)
        M[rat_to_i[string(r[rat])], state_to_j[r[state]]] = r[param]
    end

    cl = robust_clims(M; q=q)

    nrows, ncols = size(M)

    # Size so cells are square and labels fit
    W = left_pad + right_pad + cell_px * ncols
    H = top_pad + bottom_pad + cell_px * nrows
    fig = Figure(size=(W, H), figure_padding=(10, 10, 10, 10))

    ax = Axis(fig[2, 1];
        title=title,
        xlabel="State",
        ylabel="Rat",
        xticks=(1:ncols, string.(states)),
        yticks=(1:nrows, rats),
        aspect=DataAspect(),
        xgridvisible=false, ygridvisible=false,
        topspinevisible=false, rightspinevisible=false,
    )

    # IMPORTANT: plot transpose so x ↔ columns(states), y ↔ rows(rats)
    hm = CairoMakie.heatmap!(ax, 1:ncols, 1:nrows, M';
        colormap=cmap,
        colorrange=cl,
        nan_color=(:gray90, 1.0),
    )

    # Match your old yflip=true (rat #1 at top)
    CairoMakie.ylims!(ax, nrows + 0.5, 0.5)
    CairoMakie.xlims!(ax, 0.5, ncols + 0.5)

    cbgrid = GridLayout(fig[1, 1])              # a nested grid above the axis
    cb = Colorbar(cbgrid[1, 2], hm; vertical=false)

    cb.height = 18
    rowgap!(fig.layout, 6)

    # Keep figure layout sane
    rowsize!(fig.layout, 1, Auto(28))
    rowsize!(fig.layout, 2, Relative(1))
    colsize!(fig.layout, 1, Relative(1))

    return fig
end


rats_order = rats_order_from_param(ddm_params_df, :v; rat=:rat, state=:state,
                                   order_rows_by=:mean, order_rows_rev=true)

fig_v   = param_heatmap_pretty_makie(ddm_params_df, :v;   title="v (drift)", rats_order=rats_order, cell_px=26)
fig_tau = param_heatmap_pretty_makie(ddm_params_df, :tau; title="τ",       rats_order=rats_order, cell_px=26)
fig_a0  = param_heatmap_pretty_makie(ddm_params_df, :a0;  title="a₀",        rats_order=rats_order, cell_px=26)
fig_B   = param_heatmap_pretty_makie(ddm_params_df, :B;   title="B",         rats_order=rats_order, cell_px=26)

save("../results/population_v_sat.eps", fig_v)
save("../results/population_tau_sat.eps", fig_tau)
save("../results/population_a0_sat.eps", fig_a0)
save("../results/population_B_sat.eps", fig_B)

In [ ]:
function unique_trial_timeline_stylized(; tmax=4.0,
    # digital signals
    center_light_on=0.0, center_light_off=1.0,
    center_poke_on=0.60, center_poke_off=1.30,
    decision_on=2.05, reward_off=3.10,

    # flash timing
    flash_dt=0.10,
    flash_start_offset=0.10,

    # probabilistic side assignment
    p_right=0.75,
    rng=Random.default_rng(),

    # labels (top to bottom)
    labels=["center light","center poke","right flash","left flash","decision/reward"],

    # style knobs
    row_band_alpha=0.06,
    task_band_alpha=0.12,
    step_height=0.62,
    task_box_lw=1,
)
    n = length(labels)
    y = collect(n:-1:1)

    p = plot(
        xlim=(0, tmax),
        ylim=(0.5, n + 0.5),
        yticks=(1:n, reverse(labels)),
        xlabel="time in trial (s)",
        legend=false,
        framestyle=:box,
        grid=false,
        color=:black,
        size=(760, 300),
    )

    function add_rect!(x1, x2, y1, y2; fillα=0.1)
        xs = [x1, x2, x2, x1]
        ys = [y1, y1, y2, y2]
        plot!(p, Shape(xs, ys);
              seriestype=:shape,
              linealpha=0,          # no border
              fillalpha=fillα,
              color=:black)   
        return nothing
    end

    # Task period band + outline
    x1 = clamp(center_poke_on, 0, tmax)
    x2 = clamp(decision_on, 0, tmax)
    if x2 > x1
        add_rect!(x1, x2, 0.5, n + 0.5; fillα=task_band_alpha)
    end

    # Start marker at t=0
    plot!(p, [0.0, 0.0], [0.5, n + 0.5]; lw=1, color=:black)

    # helpers
    function add_step!(t_on, t_off, yrow; height=step_height, lw=2)
        t_on  = clamp(t_on,  0, tmax)
        t_off = clamp(t_off, 0, tmax)
        t = [0.0, t_on, t_off, tmax]
        v = ([0.0, 0.0, 1.0, 0.0] .* height) .+ (yrow - height/2)
        plot!(p, t, v; seriestype=:steppre, lw=lw, color=:black)
        return nothing
    end

    function add_tick!(t, yrow; tick=0.30, cap=0.018, lw=2)
        plot!(p, [t, t], [yrow - tick, yrow + tick]; lw=lw, color=:black)
        return nothing
    end

    function add_ticks!(times, yrow; tick=0.30, cap=0.018, lw=2)
        for t in times
            (0.0 ≤ t ≤ tmax) && add_tick!(t, yrow; tick=tick, cap=cap, lw=lw)
        end
        return nothing
    end

    # flash times: start 100 ms after center poke, every 100 ms until decision
    flash_start = center_poke_on + flash_start_offset
    flash_end   = decision_on
    flash_times = (flash_end > flash_start) ? collect(flash_start:flash_dt:(flash_end - 1e-9)) : Float64[]

    # split flashes by probability
    right_flash_times = Float64[]
    left_flash_times  = Float64[]
    # add bialteral falsh at center nose poke
    push!(right_flash_times, center_poke_on)
    push!(left_flash_times, center_poke_on)
    for t in flash_times
        if rand(rng) < p_right
            push!(right_flash_times, t)
        else
            push!(left_flash_times, t)
        end
    end

    # draw rows
    add_step!(center_light_on, center_light_off, y[1]; lw=2)
    add_step!(center_poke_on,  center_poke_off,  y[2]; lw=2)

    add_ticks!(right_flash_times, y[3]; tick=0.34, cap=0.020, lw=2)
    add_ticks!(left_flash_times,  y[4]; tick=0.24, cap=0.014, lw=2)

    add_step!(decision_on, reward_off, y[5]; height=0.70, lw=3)

    return p
end

# Example
p = unique_trial_timeline_stylized(
    tmax=4.0,
    center_light_on=0.0, center_light_off=0.6,
    center_poke_on=0.60, center_poke_off=0.7,
    decision_on=2.10, reward_off=3.20,
    p_right=0.75,
    rng=MersenneTwister(1234),
)
display(p)
savefig(p, "../results/stylized_trial_timeline.svg")
